# What are its radius and mass? Fit the transits and the velocities together

[Open this notebook in Colab](https://colab.research.google.com/github/Rakiura-Inc/limits-of-habitability/blob/main/notebooks/tess_02_radius_and_mass.ipynb)

In the bootcamp you measured a planet's radius from its dip and its mass from its star's wobble, separately, with simple models. Here you fit one model to both at once: up to eight TESS sectors and the survey's velocities share the same orbit. Velocities taken during a transit are left out, as in notebook 4. It also fits the full shape of the transit, so the radius no longer comes out too big or too small.

| Part | What you do |
|---|---|
| 1 | Fit the transits and velocities together, and compare with the published radius and mass |
| 2 | Re-run with another catalogue's values for the star, and see what moves |

Run the cells in order (Shift+Enter). Boxes headed **Write down** ask you questions. Answer each one in the cell underneath by replacing its `...`, then run that cell: the notebook keeps your answer. Each part ends with a cell that puts your answers into a single table, which is what you hand in. Each fit takes a while; the worked example prints how long it took. Keep the tab open while it runs.

In [ ]:
# Setup. Run this first. In Colab it installs what the notebook needs.
# If Colab asks you to restart the session afterwards, restart it and run the notebook from the top.
import importlib.util

# habtrain: a small package written for these notebooks. It downloads the data, draws some of the
# plots and records your answers. The science calls are in the cells, where you can read them.
if importlib.util.find_spec("habtrain") is None or importlib.util.find_spec("batman") is None:
    %pip install -q "habtrain[tess] @ git+https://github.com/Rakiura-Inc/limits-of-habitability@main"

# rakiura: the optional helper that connects this notebook to your step in Rakiura.
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"

import habtrain as ht  # the notebooks' package, used as ht.data, ht.fit, ht.plot and so on
import matplotlib.pyplot as plt  # plots
import numpy as np  # arrays and arithmetic

# Which notebook this is, so your hand-in records exactly what you ran.
NOTEBOOK = "notebooks/tess_02_radius_and_mass.ipynb"
MODE = ht.portal.connect(NOTEBOOK)

> **Connect to Rakiura (optional)**
>
> Connect, and Rakiura ticks off each part as you finish it, with your plots on the step. You still attach your answers table and press Hand in on the step yourself. Skip it and everything still works: you attach the saved files on the step.
>
> To connect, run the cell below, open the link it prints, check the code matches the one shown, and click Allow. In Colab you do this once per session. To skip the click next time, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; your key is in Rakiura under Settings > Notebook key).

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = ht.portal.connect(NOTEBOOK, sign_in=True)

## Part 1: Fit the transits and velocities together

**What the model adds.** Three things the bootcamp's simple steps left out:
- **The impact parameter:** how far from the middle of the star the planet crosses, from 0 (straight across the middle) to 1 (grazing the edge). A planet crossing near the edge makes a shorter, shallower dip.
- **Limb darkening:** the star's edge is dimmer than its middle, so the dip has rounded shoulders rather than a flat bottom.
- **The star's mass and radius,** taken from published values with their uncertainties. Together with the period they set how fast the planet crosses, which ties the transit's length to the impact parameter.

**Times.** TESS times are BTJD, which is the full Barycentric Julian Date minus 2457000. The survey's velocities come as full BJD. The notebook converts everything to BTJD; mixing the two silently ruins a fit.

In [ ]:
# Every velocity the survey released, once, for both parts.
rvs = ht.data.load_rvs()

# Worked example: TOI-669 b. Its orbit, transit length
# and star (mass and radius with uncertainties).
orbit = ht.data.orbit("TOI-669 b")
transit_hours = ht.data.transit_duration("TOI-669 b")
star = ht.data.star("TOI-669 b")

# Download up to 8 TESS sectors, flatten each with its
# transits hidden, and keep the data near transit.
toi_669 = ht.joint.prepare("TOI-669 b", rvs, orbit, transit_hours)
print(
    f"Sectors {toi_669.sectors}: {len(toi_669.times_btjd)} light-curve points near transit, "
    f"{len(toi_669.rv_ms)} velocities "
    f"({toi_669.rv_in_transit} taken during a transit are left out, as in notebook 4)"
)

### The model, written out

The cells below write the joint model the way the exoplanet case study [Fitting TESS data and radial velocities together](https://gallery.exoplanet.codes/tutorials/joint/) lays it out (Dan Foreman-Mackey, MIT licence, kept in `LICENSES/exoplanet-case-studies.txt`): the parameters and what each may take, the transit model, the velocity model, the likelihood with the star's priors, then the sampler. That case study needs PyMC3, which no longer installs, so here the transit comes from batman ([Kreidberg 2015](https://arxiv.org/abs/1507.08285)) and the sampling from emcee, as in notebook 4. Read the limits in the first cell: Part 2 asks whether the answer depends on what you assume.

In [ ]:
# The joint fit's parameters, in the order emcee sees them. After these come one velocity offset
# (m/s) for each telescope, then one log jitter for each telescope.
SHARED_PARAMETERS = [
    "t0",  # a mid-transit time near the middle of the TESS data (BTJD)
    "P",  # the period (days)
    "ratio",  # planet radius / star radius
    "impact",  # how far from the star's middle the planet crosses: 0 is the middle, 1 the edge
    "Mstar",  # the star's mass (solar masses)
    "Rstar",  # the star's radius (solar radii)
    "q1",  # limb darkening, the two numbers of Kipping 2013 (each from 0 to 1)
    "q2",
    "flux offset",  # the light curve's zero point (parts per thousand)
    "log jitter lc",  # extra scatter in the light curve, as its log
    "K",  # the wobble (m/s)
]
# Each parameter's position in that list, so the code can say params[POSITION["K"]].
POSITION = {name: number for number, name in enumerate(SHARED_PARAMETERS)}
N_SHARED = len(SHARED_PARAMETERS)

In [ ]:
# What the fit allows. Outside these limits an orbit has zero probability.
MAX_RATIO = 0.3  # a planet up to 30% of its star's radius
MAX_K_MS = 500.0
MIN_STAR_MASS_SUN = 0.1
MIN_STAR_RADIUS_SUN = 0.1
T0_WINDOW_DAYS = 0.1  # how far the transit time may move
P_WINDOW_DAYS = 0.01  # how far the period may move from the published one
LOG_JITTER_LC_RANGE = (-10, 3)
MAX_LOG_JITTER_RV = 5

In [ ]:
import batman


def batman_parameters(t0, period_days):
    """batman's parameters for a circular orbit. Size, tilt and limb darkening are placeholders."""
    params = batman.TransitParams()
    params.t0 = t0
    params.per = period_days
    params.rp = 0.03
    params.a = 10.0
    params.inc = 89.0
    params.ecc = 0.0  # a circular orbit
    params.w = 90.0  # the angle of periastron; for a circle any value gives the same transit
    params.limb_dark = "quadratic"
    params.u = [0.4, 0.2]
    return params


def transit_model(batman_params, times_btjd):
    """batman's model at these times, averaged over each 2-minute exposure (3 points in each)."""
    return batman.TransitModel(
        batman_params, times_btjd, supersample_factor=3, exp_time=ht.constants.TWO_MINUTES_DAYS
    )

In [ ]:
G, DAY = ht.constants.G, ht.constants.DAY
M_SUN, R_SUN, R_EARTH = ht.constants.M_SUN, ht.constants.R_SUN, ht.constants.R_EARTH


def days_from_transit(times_btjd, t0, period_days):
    """Days from the nearest mid-transit, for each time."""
    return (times_btjd - t0 + 0.5 * period_days) % period_days - 0.5 * period_days


def orbit_size(period_days, star_mass_sun, star_radius_sun):
    """The orbit's semi-major axis in star radii, from Kepler's third law."""
    a_cubed_m3 = G * star_mass_sun * M_SUN * (period_days * DAY) ** 2 / (4 * np.pi**2)
    semi_major_axis_m = a_cubed_m3 ** (1 / 3)
    return semi_major_axis_m / (star_radius_sun * R_SUN)

In [ ]:
def set_orbit(batman_params, params):
    """Put one set of fitted parameters into batman's: the orbit's timing, size and tilt."""
    t0, period, ratio, impact, star_mass, star_radius, q1, q2 = params[:8]
    a_over_r = orbit_size(period, star_mass, star_radius)
    batman_params.t0 = t0
    batman_params.per = period
    batman_params.rp = ratio
    batman_params.a = a_over_r
    batman_params.inc = np.degrees(np.arccos(min(impact / a_over_r, 1.0)))  # impact = a cos(tilt)
    batman_params.u = [2 * np.sqrt(q1) * q2, np.sqrt(q1) * (1 - 2 * q2)]  # Kipping 2013 to batman's

In [ ]:
def allowed(params, setup):
    """True for an orbit inside the limits above. Each check says what it rules out."""
    shared = params[:N_SHARED]
    t0, period, ratio, impact, star_mass, star_radius, q1, q2, _, log_jitter_lc, K = shared
    log_jitters_rv = params[N_SHARED + len(setup["telescopes"]) :]
    checks = [
        0 < ratio < MAX_RATIO,
        0 <= impact < 1 + ratio,  # the planet crosses at least the edge of the star
        0 < q1 < 1 and 0 < q2 < 1,  # any q1, q2 in 0 to 1 is a physical limb darkening
        0 <= K < MAX_K_MS,
        star_mass > MIN_STAR_MASS_SUN and star_radius > MIN_STAR_RADIUS_SUN,
        abs(t0 - setup["t_ref"]) < T0_WINDOW_DAYS,
        abs(period - setup["data"].period_days) < P_WINDOW_DAYS,
        LOG_JITTER_LC_RANGE[0] < log_jitter_lc < LOG_JITTER_LC_RANGE[1],
        np.all(np.abs(log_jitters_rv) < MAX_LOG_JITTER_RV),
    ]
    return all(checks)

In [ ]:
def log_likelihood(params, setup):
    """How well the transit and the velocities fit, each point weighed by its error and jitter."""
    data, telescope = setup["data"], setup["telescope_number"]
    n_telescopes = len(setup["telescopes"])
    t0, period, K = params[POSITION["t0"]], params[POSITION["P"]], params[POSITION["K"]]
    log_jitter_lc = params[POSITION["log jitter lc"]]
    offsets = params[N_SHARED : N_SHARED + n_telescopes]
    log_jitters_rv = params[N_SHARED + n_telescopes :]
    set_orbit(setup["batman_params"], params)
    brightness = setup["batman_model"].light_curve(setup["batman_params"])  # 1 is the star's usual
    flux_model = (brightness - 1) * 1e3 + params[POSITION["flux offset"]]
    rv_model = -K * np.sin(2 * np.pi * (data.rv_times_btjd - t0) / period) + offsets[telescope]
    variance_lc = data.flux_err_ppt**2 + np.exp(2 * log_jitter_lc)
    variance_rv = data.rv_err_ms**2 + np.exp(2 * log_jitters_rv[telescope])
    transit = -0.5 * np.sum((data.flux_ppt - flux_model) ** 2 / variance_lc + np.log(variance_lc))
    velocities = -0.5 * np.sum((data.rv_ms - rv_model) ** 2 / variance_rv + np.log(variance_rv))
    return transit + velocities

In [ ]:
def log_probability(params, setup):
    """The log posterior: the likelihood, plus priors on the star's published mass and radius."""
    if not allowed(params, setup):
        return -np.inf
    # each a value and uncertainty
    mass, radius = setup["published_mass"], setup["published_radius"]
    star_mass, star_radius = params[POSITION["Mstar"]], params[POSITION["Rstar"]]
    log_prior_mass = -0.5 * ((star_mass - mass.value) / mass.uncertainty) ** 2
    log_prior_radius = -0.5 * ((star_radius - radius.value) / radius.uncertainty) ** 2
    return log_likelihood(params, setup) + log_prior_mass + log_prior_radius

In [ ]:
def set_up_fit(data, published_mass, published_radius):
    """Everything the log probability needs for one planet, worked out once.

    published_mass and published_radius are the star's (value, uncertainty), in solar units:
    they enter as priors.
    """
    # A transit time in the middle of the TESS data, so the period and transit time are not tangled.
    transits_to_middle = np.round(
        (np.median(data.times_btjd) - data.transit_btjd) / data.period_days
    )
    t_ref = data.transit_btjd + transits_to_middle * data.period_days
    telescopes = sorted(set(data.telescope))
    batman_params = batman_parameters(t_ref, data.period_days)
    return {
        "data": data,
        "t_ref": t_ref,
        "telescopes": telescopes,
        "telescope_number": np.array([telescopes.index(name) for name in data.telescope]),
        "published_mass": published_mass,
        "published_radius": published_radius,
        "batman_params": batman_params,
        "batman_model": transit_model(batman_params, data.times_btjd),
    }

In [ ]:
def dip_depth(setup, transit_hours):
    """How deep the dip is, as a fraction of the star's light: the median of its middle half."""
    data = setup["data"]
    days_from_mid = days_from_transit(data.times_btjd, setup["t_ref"], data.period_days)
    in_dip = np.abs(days_from_mid) < 0.25 * transit_hours / 24
    return -np.median(data.flux_ppt[in_dip]) / 1e3  # parts per thousand to a fraction


def least_squares_velocities(setup):
    """K and each telescope's offset from a straight least-squares fit, as in notebook 4."""
    data = setup["data"]
    phase = 2 * np.pi * (data.rv_times_btjd - setup["t_ref"]) / data.period_days
    # One column of 1s and 0s for each telescope: 1 where the velocity is that telescope's.
    numbers = range(len(setup["telescopes"]))
    offset_columns = [(setup["telescope_number"] == number).astype(float) for number in numbers]
    columns = [np.sin(phase), np.cos(phase), *offset_columns]
    coefficients = np.linalg.lstsq(np.column_stack(columns), data.rv_ms, rcond=None)[0]
    return max(abs(coefficients[0]), 0.5), coefficients[2:]

In [ ]:
def starting_point(setup, transit_hours):
    """Where the walkers start, and how far they spread, for every parameter in order."""
    mass, radius = setup["published_mass"], setup["published_radius"]
    depth = dip_depth(setup, transit_hours)
    K, offsets = least_squares_velocities(setup)
    shared = [  # (start, spread)
        (setup["t_ref"], 1e-3),
        (setup["data"].period_days, 1e-6),
        (np.sqrt(max(depth, 1e-4)), 1e-3),  # ratio, from depth = ratio squared
        (0.3, 0.05),  # impact
        (mass.value, mass.uncertainty / 3),
        (radius.value, radius.uncertainty / 3),
        (0.3, 0.05),  # q1
        (0.3, 0.05),  # q2
        (0.0, 0.01),  # flux offset
        (-2.0, 0.1),  # log jitter lc: e^-2, 0.14 parts per thousand
        (K, 0.5),
    ]
    # Each telescope: its least-squares offset, and a log jitter of 1 (e^1, about 2.7 m/s).
    per_telescope = [(offset, 0.5) for offset in offsets] + [(1.0, 0.1)] * len(offsets)
    values = np.array([value for value, _ in shared + per_telescope])
    spreads = np.array([spread for _, spread in shared + per_telescope])
    return values, spreads

In [ ]:
import time

import emcee

N_WALKERS = 48


def run_joint_fit(setup, transit_hours, n_steps):
    """Start the walkers near the data's own answer and run emcee for n_steps.

    Returns emcee's sampler, which holds the chain.
    """
    start, spreads = starting_point(setup, transit_hours)
    # the same start every run, so your numbers match the answers
    rng = np.random.default_rng(seed=1)
    walker_starts = start + spreads * rng.standard_normal((N_WALKERS, len(start)))
    impact = POSITION["impact"]
    # the impact parameter is never negative
    walker_starts[:, impact] = np.abs(walker_starts[:, impact])
    np.random.seed(1)  # and the same emcee steps
    sampler = emcee.EnsembleSampler(N_WALKERS, len(start), log_probability, args=(setup,))
    # The parameters live on very different scales (a transit time near 2000 days, a period known to
    # a millionth of a day), which trips emcee's start check although the walkers are independent.
    sampler.run_mcmc(walker_starts, n_steps, progress=True, skip_initial_state_check=True)
    return sampler

In [ ]:
M_EARTH = ht.constants.M_EARTH


def mass_from_wobble(K, period_days, star_mass_sun):
    """Notebook 4's formula: the planet's mass in Earth masses, for an orbit seen edge on.

    The fit's mass divides this by the sine of the orbit's tilt, from the impact parameter.
    """
    orbit_factor = (period_days * DAY / (2 * np.pi * G)) ** (1 / 3)
    return K * orbit_factor * (star_mass_sun * M_SUN) ** (2 / 3) / M_EARTH

In [ ]:
def joint_results(sampler, setup):
    """The radius, mass, K and impact parameter from the second half of the chain.

    Every 10th step is kept: steps next to each other are nearly the same.
    """
    names = SHARED_PARAMETERS + [f"offset {name}" for name in setup["telescopes"]]
    names += [f"log jitter {name}" for name in setup["telescopes"]]
    # Notebook 4's steps: emcee's get_chain with the
    # burn-in dropped, and the 50-autocorrelation test.
    run = ht.fit.summarise(sampler, names, burn_in=sampler.iteration // 2, thin=10)
    samples = run.samples
    a_over_r = orbit_size(samples.P, samples.Mstar, samples.Rstar)
    sin_tilt = np.sqrt(1 - np.clip(samples.impact / a_over_r, 0, 1) ** 2)
    radius_earth = samples.ratio * samples.Rstar * R_SUN / R_EARTH
    mass_earth = mass_from_wobble(samples.K, samples.P, samples.Mstar) / sin_tilt
    return {
        "radius": ht.fit.estimate(radius_earth),
        "mass": ht.fit.estimate(mass_earth),
        "K": ht.fit.estimate(samples.K),
        "impact": ht.fit.estimate(samples.impact),
        "run": run,
    }

In [ ]:
def as_text(estimate, unit):
    """An estimate the way papers write it: 2.59 (+0.13, -0.11) Earth radii."""
    return f"{estimate.median:.2f} (+{estimate.plus:.2f}, -{estimate.minus:.2f}) {unit}"


def print_results(results, minutes):
    run = results["run"]
    print("Radius:", as_text(results["radius"], "Earth radii"))
    print("Mass:  ", as_text(results["mass"], "Earth masses"))
    print(
        "K:     ",
        as_text(results["K"], "m/s"),
        "  Impact parameter:",
        as_text(results["impact"], ""),
    )
    print(
        f"Converged: {run.converged} "
        f"(autocorrelation {run.autocorr_steps:.0f} steps, run {run.steps}). "
        f"The fit took {minutes:.0f} minutes."
    )

In [ ]:
def transit_curve(middle, period_days, span_days, n_points=600):
    """The fitted transit's brightness, in parts per thousand, on a fine grid of days."""
    days_grid = np.linspace(-span_days, span_days, n_points)
    centred = middle.copy()
    centred[POSITION["t0"]] = 0.0  # the model with the transit at time 0
    curve_params = batman_parameters(0.0, period_days)
    set_orbit(curve_params, centred)
    curve_flux = transit_model(curve_params, days_grid).light_curve(curve_params)
    return days_grid, (curve_flux - 1) * 1e3 + middle[POSITION["flux offset"]]

In [ ]:
def fitted_curves(setup, results):
    """The fitted curves at the middle of the posterior, with the data, for the plot."""
    data = setup["data"]
    middle = np.median(results["run"].samples.to_numpy(), axis=0)  # each parameter's middle value
    t0, period, K = middle[POSITION["t0"]], middle[POSITION["P"]], middle[POSITION["K"]]
    offsets = middle[N_SHARED : N_SHARED + len(setup["telescopes"])]
    days_from_mid = days_from_transit(data.times_btjd, t0, period)
    days_grid, flux_curve = transit_curve(middle, period, np.abs(days_from_mid).max())
    phase_grid = np.linspace(-0.5, 0.5, 600)
    return {
        "hours": days_grid * 24,
        "flux": flux_curve,
        "data_hours": days_from_mid * 24,
        "data_flux": data.flux_ppt,
        "phase": phase_grid,
        "rv": -K * np.sin(2 * np.pi * phase_grid),
        "data_phase": ((data.rv_times_btjd - t0) / period + 0.5) % 1 - 0.5,
        "data_rv": data.rv_ms - offsets[setup["telescope_number"]],
        "data_rv_err": data.rv_err_ms,
        "telescope": data.telescope,
    }

In [ ]:
# Fit transit and velocities together. The star's published mass and radius enter as priors.
# A few minutes in Colab. The optional cell further down carries on to the full 20,000 steps.
N_STEPS = 3000
FULL_FIT_STEPS = 20000  # the optional full fit below, and MY_STEPS for your planet

setup_669 = set_up_fit(toi_669, star["mass"], star["radius"])
started = time.time()
sampler_669 = run_joint_fit(setup_669, transit_hours, N_STEPS)
minutes_669 = (time.time() - started) / 60

In [ ]:
results_669 = joint_results(sampler_669, setup_669)
print_results(results_669, minutes_669)

# The fitted transit and velocity curves through the data.
fig = ht.plot.joint(
    fitted_curves(setup_669, results_669), "TOI-669 b: one orbit, two kinds of data"
)
plt.show()

The published values for TOI-669 b are a radius of 2.586 (+0.13, -0.11) Earth radii and a mass of 10.0 ± 1.4 Earth masses ([Polanski et al. 2024](https://arxiv.org/abs/2405.14786), with the radius from the transit fits of [MacDougall et al. 2023](https://doi.org/10.3847/1538-3881/acd557)). The fit lands on both.

**Converged, and the full fit.** emcee's test asks for a chain 50 autocorrelation times long. Here one autocorrelation time is about 300 steps (a short run's estimate comes out lower), so only a fit of about 15,000 steps or more passes, and the 3,000-step fit above says False. Its middle values already sit close to the full fit's; its ranges are a little rough. To run the full fit, set `RUN_FULL_FIT = True` below: the same walkers carry on to 20,000 steps, about 15 minutes more in Colab.

Some planets need longer still. TOI-1248 b, with eight sectors, still says False at 20,000 steps: its autocorrelation time is about 550 steps, so about 28,000 steps would pass. For a planet like that, False alone does not mean the fit failed. Compare the middle values of the 3,000-step and the 20,000-step fits: if they agree within their ranges, the answer is steady, though its ranges are rough. Say which you saw in `LOOKS_RIGHT`.

In [ ]:
RUN_FULL_FIT = False  # True carries the same walkers on to FULL_FIT_STEPS (20,000) in all

if RUN_FULL_FIT:
    started = time.time()
    # None: the walkers carry on from where they are
    sampler_669.run_mcmc(None, FULL_FIT_STEPS - N_STEPS, progress=True)
    minutes_669 += (time.time() - started) / 60
    results_669 = joint_results(sampler_669, setup_669)
    print_results(results_669, minutes_669)

### Your planet

Fit the planet you vetted in the previous notebook. Choose it from the map. Your planet's published radius and mass are on its NASA Exoplanet Archive page: search its name at [exoplanetarchive.ipac.caltech.edu](https://exoplanetarchive.ipac.caltech.edu).

In [ ]:
# The planets you can fit, and a menu that circles your choice on the map.
# TOI-669 b, the worked example, is left out.
planets = ht.data.planets()
planets = planets[planets.planet != "TOI-669 b"].reset_index(drop=True)
fig = ht.plot.planet_map(planets)
plt.show()
# pick_seed: "Pick one for me" gives the same planet as in tess_01.
choice = ht.choose.planet_menu(planets, pick_seed=ht.choose.TESS_PICK_SEED)

> **Write down: your planet's radius and mass**
>
> - Choose your planet in the menu above. If the menu does not appear, put its number from the map in `MY_PICK`.
> - Run the fit. The cell prints the radius, mass, K and impact parameter, and whether the fit converged.
> - In one or two lines: did it converge, and does anything in the plots look wrong? Put it in `LOOKS_RIGHT`.
>
> **Where to look:** the printed results and the two panels of the plot. A fit that looks right has its curve through the middle of the folded dip and through the folded velocities. An impact parameter near 1 means the planet crosses near the star's edge, which can make it look bigger. For Converged, see the note under the worked example.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
MY_PICK = ...  # only if the menu did not appear: your planet's number on the map
MY_STEPS = N_STEPS  # FULL_FIT_STEPS for a full fit, if Converged says False (15 minutes or more)

# Your planet: what you chose in the menu, else the number you typed, else one picked for you.
planet = ht.choose.resolve(choice, MY_PICK, table=planets, name_column="planet", what="planet")
star = ht.data.star(planet)
my_transit_hours = ht.data.transit_duration(planet)

In [ ]:
# Prepare and fit, as in the worked example (it takes about as long, or longer with more sectors).
mine = ht.joint.prepare(planet, rvs, ht.data.orbit(planet), my_transit_hours)
my_setup = set_up_fit(mine, star["mass"], star["radius"])
my_sampler = run_joint_fit(my_setup, my_transit_hours, MY_STEPS)
my_results = joint_results(my_sampler, my_setup)

In [ ]:
radius_text = as_text(my_results["radius"], "Earth radii")
mass_text = as_text(my_results["mass"], "Earth masses")
impact_text = as_text(my_results["impact"], "")
my_converged = my_results["run"].converged
print(
    f"Sectors {mine.sectors}; {len(mine.rv_ms)} velocities "
    f"({mine.rv_in_transit} taken during a transit are left out)"
)
print("Radius:", radius_text, "| Mass:", mass_text)
print("Impact parameter:", impact_text, f"| Converged: {my_converged}")
if not my_converged and MY_STEPS < FULL_FIT_STEPS:
    print("Not converged: set MY_STEPS = FULL_FIT_STEPS two cells up, and run it and these again.")
elif not my_converged:
    print("Still not converged. Some planets, such as TOI-1248 b, need more steps than this.")
    print("If the radius and mass sit inside your 3,000-step fit's ranges, say so in LOOKS_RIGHT.")
my_fig = ht.plot.joint(
    fitted_curves(my_setup, my_results), f"{planet}: one orbit, two kinds of data"
)
ht.record.keep_figure("joint-fit", "fit", my_fig)
plt.show()

In [ ]:
LOOKS_RIGHT = ...  # one or two lines, in quotes: did it converge, and does anything look wrong?

ht.record.save("my_planet", planet, "your planet")
ht.record.save("radius", radius_text, "radius")
ht.record.save("mass", mass_text, "mass")
ht.record.save("impact", impact_text, "impact parameter")
ht.record.save("converged", str(my_converged), "whether the fit converged")
ht.record.save("looks_right", LOOKS_RIGHT, "whether the fit looks right")

### Hand in Part 1

Run the cell below. It shows your Part 1 answers in one table. Check them, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `joint-fit`, and download them.

In [ ]:
# Your Part 1 answers and the fit plot, saved as your hand-in.
ht.record.handin("joint-fit", ["my_planet", "radius", "mass", "impact", "converged", "looks_right"])

## Part 2: Does the answer depend on what you assume?

The radius depends on the star's radius, and both depend on what was assumed. So far the fit used the star's mass and radius from the NASA Exoplanet Archive, which for these stars come from the published papers, where the survey measured each star from its own spectra ([MacDougall et al. 2023](https://doi.org/10.3847/1538-3881/acd557)). Here you swap them for the TESS Input Catalog's values (TIC v8.2), which come from the star's colours and its Gaia distance, with no spectrum ([Stassun et al. 2019](https://arxiv.org/abs/1905.10694)), and fit again. In a new session, run Part 1 first.

**Predict first**

Will your planet's radius move by more or less than its own uncertainty?

In [ ]:
# Your prediction, before the second fit.
MY_PREDICTION = ...  # "more", "less" or "unsure"
ht.record.predict("star_values", MY_PREDICTION)

In [ ]:
# Part 2 refits the planet you fitted in Part 1, with Part 1's data (mine, star), so run Part 1
# first. This line only checks that it ran in this session.
ht.record.answer(
    "my_planet", missing="Run Part 1 first: Part 2 refits the planet you fitted there."
)

# This only runs once you have made your prediction above.
if ht.record.revealed("star_values"):
    # The star's mass and radius from the TESS Input Catalog, then the same fit again.
    tic_star = ht.vet.tic_star(star["tic"])
    # star["mass"] and tic_star["mass"] each hold a value and its uncertainty, in solar units.
    for source, values in (("Archive star:", star), ("TIC star:    ", tic_star)):
        mass, radius = values["mass"], values["radius"]
        print(
            f"{source} mass {mass.value:.3f} ± {mass.uncertainty:.3f}, "
            f"radius {radius.value:.3f} ± {radius.uncertainty:.3f}"
        )
    tic_setup = set_up_fit(mine, tic_star["mass"], tic_star["radius"])
    tic_sampler = run_joint_fit(tic_setup, my_transit_hours, MY_STEPS)
    tic_results = joint_results(tic_sampler, tic_setup)
    print("Radius with archive star:", as_text(my_results["radius"], "Earth radii"))
    print("Radius with TIC star:    ", as_text(tic_results["radius"], "Earth radii"))
    print("Mass with TIC star:      ", as_text(tic_results["mass"], "Earth masses"))

> **Write down: what moves when the star changes**
>
> - How far did the radius move, compared with its uncertainty? Put it in `RADIUS_MOVED`.
> - Which star mass and radius would you trust more, the archive's or the TIC's, and why? One or two lines. Put it in `TRUST`.
>
> **Where to look:** the two fits printed above.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
RADIUS_MOVED = ...  # one line, in quotes
TRUST = ...  # one or two lines, in quotes

ht.record.save("radius_moved", RADIUS_MOVED, "how far the radius moved with the TIC star")
ht.record.save("trust", TRUST, "which value you would trust, and why")

**Worth knowing**

The star is one assumption; the orbit's shape is another. This model assumes a circular orbit. A planet whose transit is shorter than a circular orbit allows has to cross near the star's edge in this model, and that can make it look bigger. Fitting an eccentric orbit is beyond this notebook. If you think your planet needs one, say so in your hand-in and bring it to the office hour.

### Hand in Part 2

Run the cell below, check the table, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `star-assumption`, and download them.

In [ ]:
# Your Part 2 answers, saved as your hand-in.
ht.record.handin("star-assumption", ["radius_moved", "trust"])